# E6 - Inventario dataset axial Al-Kafri/Sudirman

Este notebook realiza un inventario controlado del dataset complementario Al-Kafri/Sudirman para evaluar su utilidad como fuente axial nativa de RM lumbar.

Alcance:

- No entrena modelos.
- No crea baseline.
- No modifica SPIDER.
- No sube datos al repositorio.
- No borra ZIPs originales.
- Solo descomprime dentro de `/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/`.
- Genera CSVs, figuras, JSON y conclusion tecnica para decidir el preprocesamiento axial del notebook 13.

In [13]:
# Dependencias para Google Colab.
try:
    import pydicom  # noqa: F401
except Exception:
    %pip install -q pydicom

In [14]:
import json
import os
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy.io import whosmat
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)

## Montaje de Drive y rutas

In [15]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
RAW_ZIPS = ALKAFRI_ROOT / "raw_zips"
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
INVENTORY_ROOT = ALKAFRI_ROOT / "inventory"
PROCESSED_ROOT = ALKAFRI_ROOT / "processed"
RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

FORCE_EXTRACT = False
EXTRACT_NESTED_ZIPS = True
FORCE_EXTRACT_NESTED = False
NESTED_EXTRACTED_ROOT = EXTRACTED_ROOT / "_nested"

for path in [RAW_ZIPS, EXTRACTED_ROOT, NESTED_EXTRACTED_ROOT, INVENTORY_ROOT, PROCESSED_ROOT, RESULTS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

if not ALKAFRI_ROOT.exists():
    raise FileNotFoundError(f"No existe ALKAFRI_ROOT: {ALKAFRI_ROOT}")

EXPECTED_ZIPS = {
    "k57fr854j2-2.zip": "main_dataset",
    "Label Image Ground Truth Data for Lumbar Spine MRI Dataset.zip": "ground_truth",
    "Source_Code.zip": "source_code",
    "Radiologists Notes for Lumbar Spine MRI Dataset.zip": "radiologist_notes",
}

print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("RAW_ZIPS:", RAW_ZIPS)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("FORCE_EXTRACT:", FORCE_EXTRACT)
print("EXTRACT_NESTED_ZIPS:", EXTRACT_NESTED_ZIPS)
print("NESTED_EXTRACTED_ROOT:", NESTED_EXTRACTED_ROOT)

ALKAFRI_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
RAW_ZIPS: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips
EXTRACTED_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted
FORCE_EXTRACT: False
EXTRACT_NESTED_ZIPS: True
NESTED_EXTRACTED_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested


## Verificacion de ZIPs

In [17]:
def human_size(num_bytes):
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.2f} {unit}"
        value /= 1024


zip_paths = sorted(RAW_ZIPS.glob("*.zip"))
zip_rows = []
for path in zip_paths:
    zip_rows.append({
        "zip_name": path.name,
        "zip_path": str(path),
        "expected": path.name in EXPECTED_ZIPS,
        "target_folder": EXPECTED_ZIPS.get(path.name, path.stem),
        "size_bytes": int(path.stat().st_size),
        "size_human": human_size(path.stat().st_size),
    })

zip_files_df = pd.DataFrame(zip_rows)
zip_files_csv_path = RESULTS_ROOT / "E6_alkafri_zip_files.csv"
zip_files_df.to_csv(zip_files_csv_path, index=False)

missing_expected = sorted(set(EXPECTED_ZIPS) - {path.name for path in zip_paths})
print("ZIPs detectados:", len(zip_paths))
print("ZIPs esperados faltantes:", missing_expected)
print("CSV ZIPs:", zip_files_csv_path)
display(zip_files_df)

if missing_expected:
    raise FileNotFoundError(f"Faltan ZIPs esperados: {missing_expected}")

ZIPs detectados: 4
ZIPs esperados faltantes: []
CSV ZIPs: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_zip_files.csv


,zip_name,zip_path,expected,target_folder,size_bytes,size_human
0,Label Image Ground Truth Data for Lumbar Spine...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,ground_truth,1671165290,1.56 GB
1,Radiologists Notes for Lumbar Spine MRI Datase...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,radiologist_notes,36027,35.18 KB
2,Source_Code.zip,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,source_code,1592126887,1.48 GB
3,k57fr854j2-2.zip,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,main_dataset,6260932186,5.83 GB


## Inventario interno de ZIPs sin descomprimir

In [18]:
KEYWORDS = [
    "axial", "sagittal", "sag", "mask", "label", "ground", "gt", "segmentation",
    "disc", "posterior", "thecal", "canal", "intervertebral", "l3", "l4", "l5", "s1",
]
TRACKED_EXTENSIONS = [".ima", ".dcm", ".png", ".jpg", ".jpeg", ".bmp", ".mat", ".csv", ".txt", ".m"]


def extension_group(name):
    ext = Path(name).suffix.lower()
    return ext if ext in TRACKED_EXTENSIONS else "other"


def keyword_flags(text):
    text_lower = str(text).lower()
    return {f"has_{kw}": bool(kw in text_lower) for kw in KEYWORDS}


zip_inventory_rows = []
for zip_path in tqdm(zip_paths, desc="Inventariando ZIPs"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        infos = [info for info in zf.infolist() if not info.is_dir()]
        names = [info.filename for info in infos]
        ext_counts = Counter(extension_group(name) for name in names)
        all_text = " ".join(names).lower()
        examples = names[:20]

        row = {
            "zip_name": zip_path.name,
            "zip_path": str(zip_path),
            "target_folder": EXPECTED_ZIPS.get(zip_path.name, zip_path.stem),
            "total_files": int(len(infos)),
            "example_internal_paths": json.dumps(examples, ensure_ascii=False),
        }
        for ext in TRACKED_EXTENSIONS + ["other"]:
            row[f"count_{ext.replace('.', '')}"] = int(ext_counts.get(ext, 0))
        row.update(keyword_flags(all_text))
        zip_inventory_rows.append(row)

zip_inventory_df = pd.DataFrame(zip_inventory_rows)
zip_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_zip_inventory.csv"
zip_inventory_df.to_csv(zip_inventory_csv_path, index=False)

print("Inventario ZIP:", zip_inventory_csv_path)
display(zip_inventory_df)

Inventariando ZIPs:   0%|          | 0/4 [00:00<?, ?it/s]

Inventario ZIP: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_zip_inventory.csv


,zip_name,zip_path,target_folder,total_files,example_internal_paths,count_ima,count_dcm,count_png,count_jpg,count_jpeg,count_bmp,count_mat,count_csv,count_txt,count_m,count_other,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1
0,Label Image Ground Truth Data for Lumbar Spine...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,ground_truth,2,"[""Label Image Ground Truth Data for Lumbar Spi...",0,0,0,0,0,0,0,0,0,0,2,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
1,Radiologists Notes for Lumbar Spine MRI Datase...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,radiologist_notes,1,"[""Radiologists Notes for Lumbar Spine MRI Data...",0,0,0,0,0,0,0,0,0,0,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,Source_Code.zip,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code,115,"[""09_Source_Code/03_Extract_Best_Slices/extrac...",0,0,6,0,0,0,32,5,13,55,4,False,False,False,True,True,True,True,True,False,False,False,False,False,False,False,False,False
3,k57fr854j2-2.zip,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,main_dataset,1,"[""MRI_Data.zip""]",0,0,0,0,0,0,0,0,0,0,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## Descompresion controlada

In [19]:
def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target_path = (destination / member.filename).resolve()
            if not str(target_path).startswith(str(destination)):
                raise RuntimeError(f"Ruta insegura dentro del ZIP: {member.filename}")
        zf.extractall(destination)


extraction_rows = []
for zip_path in tqdm(zip_paths, desc="Extrayendo ZIPs"):
    target_folder = EXPECTED_ZIPS.get(zip_path.name, zip_path.stem)
    destination = EXTRACTED_ROOT / target_folder
    destination.mkdir(parents=True, exist_ok=True)
    existing_files = [p for p in destination.rglob("*") if p.is_file()]
    skipped = bool(existing_files and not FORCE_EXTRACT)
    start = time.time()

    if skipped:
        extracted_count = len(existing_files)
        elapsed_seconds = 0.0
    else:
        if FORCE_EXTRACT and destination.exists():
            shutil.rmtree(destination)
            destination.mkdir(parents=True, exist_ok=True)
        safe_extract_zip(zip_path, destination)
        extracted_count = sum(1 for p in destination.rglob("*") if p.is_file())
        elapsed_seconds = time.time() - start

    extraction_rows.append({
        "zip_name": zip_path.name,
        "target_folder": target_folder,
        "destination": str(destination),
        "skipped_existing": skipped,
        "force_extract": FORCE_EXTRACT,
        "extracted_file_count": int(extracted_count),
        "elapsed_seconds": float(elapsed_seconds),
    })

extraction_report_df = pd.DataFrame(extraction_rows)
extraction_report_csv_path = RESULTS_ROOT / "E6_alkafri_extraction_report.csv"
extraction_report_df.to_csv(extraction_report_csv_path, index=False)

print("Reporte extraccion:", extraction_report_csv_path)
display(extraction_report_df)

Extrayendo ZIPs:   0%|          | 0/4 [00:00<?, ?it/s]

Reporte extraccion: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_extraction_report.csv


,zip_name,target_folder,destination,skipped_existing,force_extract,extracted_file_count,elapsed_seconds
0,Label Image Ground Truth Data for Lumbar Spine...,ground_truth,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,False,2,0.0
1,Radiologists Notes for Lumbar Spine MRI Datase...,radiologist_notes,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,False,1,0.0
2,Source_Code.zip,source_code,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,False,115,0.0
3,k57fr854j2-2.zip,main_dataset,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,False,1,0.0


In [20]:
nested_zip_paths = []
if EXTRACT_NESTED_ZIPS and EXTRACTED_ROOT.exists():
    nested_zip_paths = [
        p for p in EXTRACTED_ROOT.rglob("*.zip")
        if "_nested" not in p.parts
    ]


def infer_nested_source(zip_path):
    rel_parts = [part.lower() for part in Path(zip_path).relative_to(EXTRACTED_ROOT).parts]
    for source in ["main_dataset", "ground_truth", "source_code", "radiologist_notes"]:
        if source in rel_parts:
            return source
    return "unknown"


def safe_folder_name(value):
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in str(value))
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned.strip("_") or "nested_zip"


nested_extraction_rows = []
for nested_zip_path in tqdm(nested_zip_paths, desc="Extrayendo ZIPs internos"):
    inferred_source = infer_nested_source(nested_zip_path)
    stem = safe_folder_name(Path(nested_zip_path).stem)
    destination = NESTED_EXTRACTED_ROOT / f"{safe_folder_name(inferred_source)}__{stem}"
    destination.mkdir(parents=True, exist_ok=True)
    existing_files = [p for p in destination.rglob("*") if p.is_file()]
    skipped = bool(existing_files and not FORCE_EXTRACT_NESTED)
    error = None
    start = time.time()

    try:
        if skipped:
            extracted_count = len(existing_files)
            elapsed_seconds = 0.0
        else:
            if FORCE_EXTRACT_NESTED and destination.exists():
                shutil.rmtree(destination)
                destination.mkdir(parents=True, exist_ok=True)
            safe_extract_zip(nested_zip_path, destination)
            extracted_count = sum(1 for p in destination.rglob("*") if p.is_file())
            elapsed_seconds = time.time() - start
    except Exception as exc:
        extracted_count = 0
        elapsed_seconds = time.time() - start
        error = repr(exc)

    nested_extraction_rows.append({
        "nested_zip_path": str(nested_zip_path),
        "nested_zip_name": nested_zip_path.name,
        "inferred_source": inferred_source,
        "destination": str(destination),
        "skipped_existing": skipped,
        "force_extract_nested": FORCE_EXTRACT_NESTED,
        "extracted_file_count": int(extracted_count),
        "elapsed_seconds": float(elapsed_seconds),
        "error": error,
    })

nested_extraction_report_df = pd.DataFrame(nested_extraction_rows)
nested_extraction_report_csv_path = RESULTS_ROOT / "E6_alkafri_nested_extraction_report.csv"
nested_extraction_report_df.to_csv(nested_extraction_report_csv_path, index=False)

print("ZIPs internos detectados:", len(nested_zip_paths))
print("Reporte extraccion nested:", nested_extraction_report_csv_path)
display(nested_extraction_report_df)

Extrayendo ZIPs internos:   0%|          | 0/3 [00:00<?, ?it/s]

ZIPs internos detectados: 3
Reporte extraccion nested: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_nested_extraction_report.csv


,nested_zip_path,nested_zip_name,inferred_source,destination,skipped_existing,force_extract_nested,extracted_file_count,elapsed_seconds,error
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,MRI_Data.zip,main_dataset,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,True,False,17497,0.000000,None
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,Manual_Label_Data.zip,ground_truth,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,23175,567.914710,None
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,Ground_Truth_Label.zip,ground_truth,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,13905,381.717068,None


## Inventario posterior a la extraccion

In [21]:
def probable_source_from_path(path):
    path = Path(path)
    try:
        rel_parts = [part.lower() for part in path.relative_to(EXTRACTED_ROOT).parts]
    except Exception:
        rel_parts = [part.lower() for part in path.parts]
    for source in ["main_dataset", "ground_truth", "source_code", "radiologist_notes"]:
        if source in rel_parts:
            return source
    if "_nested" in rel_parts:
        for source in ["main_dataset", "ground_truth", "source_code", "radiologist_notes"]:
            if any(part.startswith(f"{source}__") for part in rel_parts):
                return source
    return "unknown"


def probable_type_from_path(path):
    path = Path(path)
    ext = path.suffix.lower() if path.suffix else "<sin_extension>"
    text = str(path).lower()
    if ext == ".ima":
        return "image_dicom_ima"
    if ext == ".dcm":
        return "image_dicom_dcm"
    if ext in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"} and any(k in text for k in ["mask", "label", "ground", "gt", "seg"]):
        return "mask_or_label_image"
    if ext in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return "image_file"
    if ext == ".mat":
        return "matlab_file"
    if ext in {".txt", ".csv", ".xlsx", ".xls", ".doc", ".docx"}:
        return "text_metadata"
    if ext == ".zip":
        return "zip_archive"
    return "unknown"


def nested_source_zip_for_path(path):
    path = Path(path)
    try:
        rel = path.relative_to(NESTED_EXTRACTED_ROOT)
    except Exception:
        return None
    if not rel.parts:
        return None
    top_folder = rel.parts[0]
    if "nested_extraction_report_df" in globals() and len(nested_extraction_report_df) > 0:
        match = nested_extraction_report_df[
            nested_extraction_report_df["destination"].astype(str).eq(str(NESTED_EXTRACTED_ROOT / top_folder))
        ]
        if len(match) > 0:
            return str(match["nested_zip_path"].iloc[0])
    return top_folder


def iter_inventory_files():
    seen = set()
    for root in [EXTRACTED_ROOT, NESTED_EXTRACTED_ROOT]:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if path.is_file():
                resolved = str(path.resolve())
                if resolved in seen:
                    continue
                seen.add(resolved)
                yield path


file_rows = []
for path in tqdm(list(iter_inventory_files()), desc="Inventario extraido"):
    rel = path.relative_to(EXTRACTED_ROOT) if str(path).startswith(str(EXTRACTED_ROOT)) else path.relative_to(NESTED_EXTRACTED_ROOT)
    ext = path.suffix.lower() if path.suffix else "<sin_extension>"
    is_nested = "_nested" in path.parts or str(path).startswith(str(NESTED_EXTRACTED_ROOT))
    row = {
        "file_path": str(path),
        "relative_path": str(rel),
        "file_name": path.name,
        "extension": ext,
        "parent_folder": path.parent.name,
        "size_bytes": int(path.stat().st_size),
        "probable_source": probable_source_from_path(path),
        "probable_type": probable_type_from_path(path),
        "is_nested": bool(is_nested),
        "nested_source_zip": nested_source_zip_for_path(path) if is_nested else None,
    }
    row.update(keyword_flags(str(rel)))
    file_rows.append(row)

extracted_inventory_df = pd.DataFrame(file_rows)
extracted_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv"
extracted_inventory_df.to_csv(extracted_inventory_csv_path, index=False)

print("Inventario extraido:", extracted_inventory_csv_path)
print("Archivos inventariados:", len(extracted_inventory_df))
display(extracted_inventory_df.head())

Inventario extraido:   0%|          | 0/54696 [00:00<?, ?it/s]

Inventario extraido: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_extracted_file_inventory.csv
Archivos inventariados: 54696


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/initialise.m,initialise.m,.m,source_code,5337,source_code,unknown,False,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/Instruction.docx,Instruction.docx,.docx,source_code,16923,source_code,text_metadata,False,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/Setup.bat,Setup.bat,.bat,source_code,173,source_code,unknown,False,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,main_dataset/MRI_Data.zip,MRI_Data.zip,.zip,main_dataset,6270535053,main_dataset,zip_archive,False,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,ground_truth/Label Image Ground Truth Data for...,Manual_Label_Data.zip,.zip,Label Image Ground Truth Data for Lumbar Spine...,637438849,ground_truth,zip_archive,False,None,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False


## Estructura de carpetas y extensiones

In [22]:
def summarize_tree(root, max_depth=3, max_entries=250):
    lines = []
    root = Path(root)
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        lines.append(f"{indent}{rel.name}{suffix}")
        if len(lines) >= max_entries:
            lines.append("...")
            break
    return "\n".join(lines)


tree_text = summarize_tree(EXTRACTED_ROOT, max_depth=3)
print(tree_text[:6000])

folder_summary_df = (
    extracted_inventory_df
    .assign(top_folder=lambda df: df["relative_path"].apply(lambda p: Path(p).parts[0] if Path(p).parts else "root"))
    .groupby(["top_folder", "probable_source"], dropna=False)
    .agg(n_files=("file_path", "count"), size_bytes=("size_bytes", "sum"))
    .reset_index()
)
folder_summary_df["size_human"] = folder_summary_df["size_bytes"].apply(human_size)

extension_summary_df = (
    extracted_inventory_df
    .groupby(["extension", "probable_type"], dropna=False)
    .agg(n_files=("file_path", "count"), size_bytes=("size_bytes", "sum"))
    .reset_index()
    .sort_values("n_files", ascending=False)
)
extension_summary_df["size_human"] = extension_summary_df["size_bytes"].apply(human_size)

folder_summary_csv_path = RESULTS_ROOT / "E6_alkafri_folder_summary.csv"
extension_summary_csv_path = RESULTS_ROOT / "E6_alkafri_extension_summary.csv"
folder_summary_df.to_csv(folder_summary_csv_path, index=False)
extension_summary_df.to_csv(extension_summary_csv_path, index=False)

print("Folder summary:", folder_summary_csv_path)
print("Extension summary:", extension_summary_csv_path)
display(folder_summary_df)
display(extension_summary_df)

_nested/
  ground_truth__Ground_Truth_Label/
    04_Intermediary_Ground_Truth_Data/
    05_Final_Ground_Truth_Data/
  ground_truth__Manual_Label_Data/
    03_Manual_Label_Data/
  main_dataset__MRI_Data/
    01_MRI_Data/
ground_truth/
  Label Image Ground Truth Data for Lumbar Spine MRI Dataset/
    Ground_Truth_Label.zip
    Manual_Label_Data.zip
main_dataset/
  MRI_Data.zip
radiologist_notes/
  Radiologists Notes for Lumbar Spine MRI Dataset/
    Radiologists Report.xlsx
source_code/
  09_Source_Code/
    03_Extract_Best_Slices/
    04_Manual_Label_Checking/
    05_Ground_Truth_Development/
    06_Semantic_Segmentation/
    07_Agreement_Analysis/
    99_External_Libraries/
  Instruction.docx
  Setup.bat
  initialise.m
Folder summary: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_folder_summary.csv
Extension summary: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_extension_summary.csv


,top_folder,probable_source,n_files,size_bytes,size_human
0,_nested,ground_truth,37080,1857172897,1.73 GB
1,_nested,main_dataset,17497,4891858350,4.56 GB
2,ground_truth,ground_truth,2,1677117727,1.56 GB
3,main_dataset,main_dataset,1,6270535053,5.84 GB
4,radiologist_notes,radiologist_notes,1,38292,37.39 KB
5,source_code,source_code,115,1658169403,1.54 GB


,extension,probable_type,n_files,size_bytes,size_human
7,.png,mask_or_label_image,29361,1057796603,1008.79 MB
3,.ima,image_dicom_ima,17497,4891858350,4.56 GB
9,.xcf,unknown,7725,799588173,762.55 MB
4,.m,unknown,55,210777,205.84 KB
5,.mat,matlab_file,32,1657664069,1.54 GB
8,.txt,text_metadata,13,2165,2.11 KB
1,.csv,text_metadata,5,25059,24.47 KB
11,.zip,zip_archive,3,7947652780,7.40 GB
0,.bat,unknown,1,173,173.00 B
2,.docx,text_metadata,1,16923,16.53 KB


## Lectura preliminar DICOM / IMA

In [23]:
print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("RAW_ZIPS:", RAW_ZIPS)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("EXTRACTED_ROOT existe:", EXTRACTED_ROOT.exists())

if EXTRACTED_ROOT.exists():
    all_files = [p for p in EXTRACTED_ROOT.rglob("*") if p.is_file()]
    print("Total archivos extraidos:", len(all_files))

    ext_counts = {}
    for p in all_files:
        ext = p.suffix.lower() if p.suffix else "<sin_extension>"
        ext_counts[ext] = ext_counts.get(ext, 0) + 1

    ext_df = pd.DataFrame(
        [{"extension": k, "n": v} for k, v in sorted(ext_counts.items(), key=lambda x: x[1], reverse=True)]
    )

    display(ext_df.head(30))

    print("\nEjemplos de archivos:")
    for p in all_files[:30]:
        print("-", p)
else:
    print("No existe EXTRACTED_ROOT")

ALKAFRI_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
RAW_ZIPS: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips
EXTRACTED_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted
EXTRACTED_ROOT existe: True
Total archivos extraidos: 54696


,extension,n
0,.png,29361
1,.ima,17497
2,.xcf,7725
3,.m,55
4,.mat,32
5,.txt,13
6,.csv,5
7,.zip,3
8,.docx,1
9,.bat,1



Ejemplos de archivos:
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/initialise.m
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/Instruction.docx
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/Setup.bat
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/main_dataset/MRI_Data.zip
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/ground_truth/Label Image Ground Truth Data for Lumbar Spine MRI Dataset/Manual_Label_Data.zip
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/ground_truth/Label Image Ground Truth Data for Lumbar Spine MRI Dataset/Ground_Truth_Label.zip
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/radiologist_notes/Radiologists Notes for Lumbar Spine MRI Dataset/Radiologists Report.xlsx
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/extract_dicom.m
- /content/drive/MyDrive/PFI_MVP/

In [24]:
DICOM_TAGS = [
    "PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription",
    "ProtocolName", "Modality", "ImageOrientationPatient", "ImagePositionPatient",
    "PixelSpacing", "SliceThickness", "Rows", "Columns", "InstanceNumber",
]
MEDICAL_DICOM_TAGS = ["Rows", "Columns", "PixelData", "PatientID", "SeriesInstanceUID", "ImageOrientationPatient"]


def safe_dcmread(path, stop_before_pixels=True):
    try:
        return pydicom.dcmread(str(path), stop_before_pixels=stop_before_pixels, force=True), None
    except Exception as exc:
        return None, repr(exc)


def get_dicom_tag(ds, tag_name):
    value = getattr(ds, tag_name, None)
    if value is None:
        return None
    try:
        if isinstance(value, pydicom.multival.MultiValue) or isinstance(value, (list, tuple)):
            return json.dumps([str(v) for v in value])
        return str(value)
    except Exception:
        return repr(value)


def has_medical_dicom_tags(ds):
    if ds is None:
        return False
    return any(hasattr(ds, tag) for tag in MEDICAL_DICOM_TAGS)


def is_plausible_no_extension_dicom(row):
    text = f"{row.get('relative_path', '')} {row.get('file_name', '')}".lower()
    if row.get("probable_source") in {"source_code", "radiologist_notes"}:
        return False
    if str(row.get("file_name", "")).lower() in {"license", "readme", "copying"}:
        return False
    if int(row.get("size_bytes", 0) or 0) < 1024:
        return False
    if "external_libraries" in text or "external" in text or "library" in text:
        return False
    return any(keyword in text for keyword in ["dicom", "mri", "image", "patient", "series"])


if len(extracted_inventory_df) > 0:
    extracted_inventory_df["extension"] = extracted_inventory_df["extension"].astype(str).str.lower()

dicom_by_extension_df = extracted_inventory_df[
    extracted_inventory_df["extension"].isin([".ima", ".dcm"])
].copy()

no_ext_candidates_df = extracted_inventory_df[
    extracted_inventory_df["extension"].isin(["<sin_extension>", "", "."])
].copy()
no_ext_candidates_df = no_ext_candidates_df[
    no_ext_candidates_df.apply(is_plausible_no_extension_dicom, axis=1)
].copy()

dicom_candidates_df = pd.concat([dicom_by_extension_df, no_ext_candidates_df], ignore_index=True).drop_duplicates("file_path")

print("Candidatos DICOM/IMA por extension:", len(dicom_by_extension_df))
print("Candidatos sin extension plausibles:", len(no_ext_candidates_df))
print("Candidatos DICOM/IMA totales:", len(dicom_candidates_df))
if len(dicom_candidates_df) > 0:
    display(dicom_candidates_df.head(20))

dicom_metadata_rows = []
dicom_reading_rows = []
valid_dicom_paths = []
for _, row_inv in tqdm(dicom_candidates_df.iterrows(), total=len(dicom_candidates_df), desc="Leyendo candidatos DICOM/IMA"):
    file_path = row_inv["file_path"]
    ds, error = safe_dcmread(file_path, stop_before_pixels=True)
    read_ok = ds is not None
    valid_medical = has_medical_dicom_tags(ds)
    discard_reason = None
    if not read_ok:
        discard_reason = "read_error"
    elif not valid_medical:
        discard_reason = "no_medical_tags"
    else:
        valid_dicom_paths.append(file_path)

    metadata_row = {
        "file_path": file_path,
        "relative_path": row_inv.get("relative_path", ""),
        "extension": row_inv.get("extension", ""),
        "read_ok": read_ok,
        "valid_medical_dicom": valid_medical,
        "read_error": error,
        "discard_reason": discard_reason,
    }
    for tag in DICOM_TAGS:
        metadata_row[tag] = get_dicom_tag(ds, tag) if ds is not None else None
    dicom_metadata_rows.append(metadata_row)
    dicom_reading_rows.append({
        "file_path": file_path,
        "relative_path": row_inv.get("relative_path", ""),
        "extension": row_inv.get("extension", ""),
        "candidate_source": "extension" if row_inv.get("extension", "") in [".ima", ".dcm"] else "plausible_no_extension",
        "read_ok": read_ok,
        "valid_medical_dicom": valid_medical,
        "discard_reason": discard_reason,
        "read_error": error,
    })

expected_metadata_columns = [
    "file_path", "relative_path", "extension", "read_ok", "valid_medical_dicom",
    "read_error", "discard_reason", *DICOM_TAGS,
]
dicom_metadata_sample_df = pd.DataFrame(dicom_metadata_rows)
for col in expected_metadata_columns:
    if col not in dicom_metadata_sample_df.columns:
        dicom_metadata_sample_df[col] = None
dicom_metadata_sample_df = dicom_metadata_sample_df[expected_metadata_columns]

dicom_reading_report_df = pd.DataFrame(dicom_reading_rows)
expected_reading_columns = [
    "file_path", "relative_path", "extension", "candidate_source",
    "read_ok", "valid_medical_dicom", "discard_reason", "read_error",
]
for col in expected_reading_columns:
    if col not in dicom_reading_report_df.columns:
        dicom_reading_report_df[col] = None
dicom_reading_report_df = dicom_reading_report_df[expected_reading_columns]

valid_dicom_df = dicom_metadata_sample_df[dicom_metadata_sample_df["valid_medical_dicom"].fillna(False)].copy()

dicom_metadata_sample_csv_path = RESULTS_ROOT / "E6_alkafri_dicom_metadata_sample.csv"
dicom_reading_report_csv_path = RESULTS_ROOT / "E6_alkafri_dicom_reading_report.csv"
dicom_metadata_sample_df.to_csv(dicom_metadata_sample_csv_path, index=False)
dicom_reading_report_df.to_csv(dicom_reading_report_csv_path, index=False)

print("Candidatos por extension:", len(dicom_by_extension_df))
print("Candidatos leidos correctamente:", int(dicom_reading_report_df["read_ok"].fillna(False).sum()) if len(dicom_reading_report_df) else 0)
print("DICOM validos con tags medicos:", len(valid_dicom_df))
print("DICOM descartados sin tags medicos:", int((dicom_reading_report_df["discard_reason"] == "no_medical_tags").sum()) if len(dicom_reading_report_df) else 0)
print("Metadata sample:", dicom_metadata_sample_csv_path)
print("Reading report:", dicom_reading_report_csv_path)
display(dicom_metadata_sample_df.head(20))

Candidatos DICOM/IMA por extension: 17497
Candidatos sin extension plausibles: 0
Candidatos DICOM/IMA totales: 17497


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_001.ima,.ima,LOCALIZER_0001,526326,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_002.ima,.ima,LOCALIZER_0001,526336,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_003.ima,.ima,LOCALIZER_0001,526332,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_004.ima,.ima,LOCALIZER_0001,526328,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_005.ima,.ima,LOCALIZER_0001,526332,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
5,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_006.ima,.ima,LOCALIZER_0001,526334,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_007.ima,.ima,LOCALIZER_0001,526332,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
7,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,LOCALIZER_0_0001_008.ima,.ima,LOCALIZER_0001,526324,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,POSDISP_[4]_0001_001.ima,.ima,POSDISP_[4]_T2_TSE_TRA_384_5001,559760,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,POSDISP_[4]_0001_002.ima,.ima,POSDISP_[4]_T2_TSE_TRA_384_5001,316466,main_dataset,image_dicom_ima,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


Leyendo candidatos DICOM/IMA:   0%|          | 0/17497 [00:00<?, ?it/s]

Candidatos por extension: 17497
Candidatos leidos correctamente: 17497
DICOM validos con tags medicos: 17496
DICOM descartados sin tags medicos: 1
Metadata sample: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_dicom_metadata_sample.csv
Reading report: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_dicom_reading_report.csv


,file_path,relative_path,extension,read_ok,valid_medical_dicom,read_error,discard_reason,PatientID,StudyInstanceUID,SeriesInstanceUID,SeriesDescription,ProtocolName,Modality,ImageOrientationPatient,ImagePositionPatient,PixelSpacing,SliceThickness,Rows,Columns,InstanceNumber
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""-12"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,1
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""0"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,2
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""12"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,3
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""-14"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,4
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""18"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,5
5,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""50"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,6
6,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""82"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,7
7,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""114"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,8
8,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.3000001603090723584...,PosDisp: [4] t2_tse_tra_384,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-174.83898824897"", ""18"", ""174.83898824897""]","[""0.68359375"", ""0.68359375""]",8,512,512,5
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.3000001603090723584...,PosDisp: [4] t2_tse_tra_384,None,MR,"[""-6.08500845E-12"", ""1"", ""-2.05013285E-10"", ""-...","[""11.12014242056"", ""-114.01582770

## Clasificacion preliminar axial/sagital

In [25]:
def orientation_candidate_from_text_and_tags(row):
    path_text = str(row.get("relative_path", "")).lower()
    series_text = str(row.get("SeriesDescription", "") or "").lower()
    protocol_text = str(row.get("ProtocolName", "") or "").lower()
    orientation = row.get("ImageOrientationPatient")

    for label, text, reason in [
        ("axial_candidate", path_text, "path_keyword"),
        ("axial_candidate", series_text, "series_description_keyword"),
        ("axial_candidate", protocol_text, "protocol_keyword"),
    ]:
        if "ax" in text or "axial" in text or "tra" in text:
            return label, reason

    for label, text, reason in [
        ("sagittal_candidate", path_text, "path_keyword"),
        ("sagittal_candidate", series_text, "series_description_keyword"),
        ("sagittal_candidate", protocol_text, "protocol_keyword"),
    ]:
        if "sag" in text or "sagittal" in text:
            return label, reason

    if orientation not in [None, "", "None"]:
        try:
            if isinstance(orientation, str):
                values = np.asarray(json.loads(orientation), dtype=float)
            else:
                values = np.asarray(orientation, dtype=float)
            row_cosine = values[:3]
            col_cosine = values[3:]
            normal = np.abs(np.cross(row_cosine, col_cosine))
            dominant = int(np.argmax(normal))
            if dominant == 2:
                return "axial_candidate", "orientation_inference"
            if dominant == 0:
                return "sagittal_candidate", "orientation_inference"
        except Exception:
            pass
    return "unknown", "unknown"


expected_columns = [
    "file_path", "relative_path", "extension", "read_ok", "valid_medical_dicom",
    "read_error", "discard_reason", *DICOM_TAGS,
    "orientation_candidate", "classification_reason",
]

orientation_rows = []
if len(valid_dicom_df) == 0:
    print("No hay DICOM/IMA validos para clasificar. Se exporta CSV vacio con columnas esperadas.")
    series_orientation_df = pd.DataFrame(columns=expected_columns)
else:
    for _, base_row in tqdm(valid_dicom_df.iterrows(), total=len(valid_dicom_df), desc="Clasificando DICOM/IMA validos"):
        base = base_row.to_dict()
        candidate, reason = orientation_candidate_from_text_and_tags(base)
        base["orientation_candidate"] = candidate
        base["classification_reason"] = reason
        orientation_rows.append(base)
    series_orientation_df = pd.DataFrame(orientation_rows)
    for col in expected_columns:
        if col not in series_orientation_df.columns:
            series_orientation_df[col] = None
    series_orientation_df = series_orientation_df[expected_columns]

series_orientation_csv_path = RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv"
series_orientation_df.to_csv(series_orientation_csv_path, index=False)

print("Orientaciones:", series_orientation_csv_path)
if len(series_orientation_df) > 0:
    display(series_orientation_df["orientation_candidate"].value_counts(dropna=False).rename_axis("candidate").reset_index(name="n"))
    display(series_orientation_df.head())
else:
    print("series_orientation_df vacio.")

Clasificando DICOM/IMA validos:   0%|          | 0/17496 [00:00<?, ?it/s]

Orientaciones: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_series_orientation_candidates.csv


,candidate,n
0,axial_candidate,8203
1,sagittal_candidate,7646
2,unknown,1647


,file_path,relative_path,extension,read_ok,valid_medical_dicom,read_error,discard_reason,PatientID,StudyInstanceUID,SeriesInstanceUID,SeriesDescription,ProtocolName,Modality,ImageOrientationPatient,ImagePositionPatient,PixelSpacing,SliceThickness,Rows,Columns,InstanceNumber,orientation_candidate,classification_reason
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""-12"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,1,sagittal_candidate,orientation_inference
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""0"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,2,sagittal_candidate,orientation_inference
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""0"", ""1"", ""0"", ""0"", ""0"", ""-1""]","[""12"", ""-125"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,3,sagittal_candidate,orientation_inference
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""-14"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,4,unknown,unknown
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,_nested/main_dataset__MRI_Data/01_MRI_Data/000...,.ima,True,True,None,None,,1.3.12.2.1107.5.2.40.50233.3000001603090708352...,1.3.12.2.1107.5.2.40.50233.2016030909181239634...,localizer,None,MR,"[""1"", ""0"", ""0"", ""0"", ""0"", ""-1""]","[""-175"", ""18"", ""175""]","[""0.68359375"", ""0.68359375""]",8,512,512,5,unknown,unknown


## Inventario de ground truth

In [26]:
GT_KEYWORDS = [
    "disc", "posterior", "thecal", "canal", "intervertebral",
    "l3", "l4", "l5", "s1", "mask", "label", "ground", "gt", "segmentation",
]


def gt_keyword_flags(text):
    text_lower = str(text).lower()
    return {f"has_{kw}": bool(kw in text_lower) for kw in GT_KEYWORDS}


gt_df = extracted_inventory_df[
    (extracted_inventory_df["probable_source"].eq("ground_truth"))
    | extracted_inventory_df["has_ground"]
    | extracted_inventory_df["has_label"]
    | extracted_inventory_df["has_mask"]
    | extracted_inventory_df["has_gt"]
].copy()

gt_rows = []
for _, row in gt_df.iterrows():
    text = f"{row['relative_path']} {row['file_name']}"
    item = row.to_dict()
    item.update(gt_keyword_flags(text))
    if row["extension"] == ".mat":
        try:
            item["mat_variables"] = json.dumps([var[0] for var in whosmat(row["file_path"])], ensure_ascii=False)
        except Exception as exc:
            item["mat_variables"] = None
            item["mat_read_error"] = repr(exc)
    gt_rows.append(item)

ground_truth_inventory_df = pd.DataFrame(gt_rows)
ground_truth_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv"
ground_truth_inventory_df.to_csv(ground_truth_inventory_csv_path, index=False)

print("Ground truth files:", len(ground_truth_inventory_df))
print("Ground truth inventory:", ground_truth_inventory_csv_path)
if len(ground_truth_inventory_df) > 0:
    display(ground_truth_inventory_df.head(30))
else:
    print("No se detectaron archivos de ground truth por fuente o keywords.")

Ground truth files: 37129
Ground truth inventory: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_ground_truth_inventory.csv


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1,mat_variables,mat_read_error
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,ground_truth/Label Image Ground Truth Data for...,Manual_Label_Data.zip,.zip,Label Image Ground Truth Data for Lumbar Spine...,637438849,ground_truth,zip_archive,False,None,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,ground_truth/Label Image Ground Truth Data for...,Ground_Truth_Label.zip,.zip,Label Image Ground Truth Data for Lumbar Spine...,1039678878,ground_truth,zip_archive,False,None,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,calc_pf.m,.m,04_Manual_Label_Checking,2679,source_code,unknown,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,check_mask_label.m,.m,04_Manual_Label_Checking,4164,source_code,unknown,False,None,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,count_png_files.m,.m,04_Manual_Label_Checking,2483,source_code,unknown,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
5,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,count_votes.m,.m,04_Manual_Label_Checking,4034,source_code,unknown,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
6,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,Figure (a).png,.png,04_Manual_Label_Checking,1796,source_code,mask_or_label_image,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
7,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,Figure (b).png,.png,04_Manual_Label_Checking,2251,source_code,mask_or_label_image,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
8,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,Figure (c).png,.png,04_Manual_Label_Checking,1383,source_code,mask_or_label_image,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/04_Manual_Label_Che...,Figure (d).png,.png,04_Manual_Label_Checking,2107,source_code,mask_or_label_image,False,None,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN


## Visualizacion preliminar de imagenes axiales nativas

In [27]:
def normalize_percentile(image, p_low=1, p_high=99):
    arr = image.astype(np.float32)
    low, high = np.percentile(arr, [p_low, p_high])
    if np.isclose(low, high):
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - low) / (high - low), 0, 1).astype(np.float32)


def read_dicom_pixels(path):
    ds = pydicom.dcmread(str(path), force=True)
    arr = ds.pixel_array.astype(np.float32)
    return ds, arr


def is_visualization_raster_candidate(row):
    name = str(row.get("file_name", ""))
    if row.get("probable_source") in {"source_code", "radiologist_notes"}:
        return False
    if name.lower().startswith("figure"):
        return False
    if "source_code" in str(row.get("relative_path", "")).lower():
        return False
    return row.get("probable_source") in {"main_dataset", "ground_truth", "unknown"}


axial_image_paths = []
visualization_errors = []
overlay_preliminary_path = None
overlay_pairing_status = "No se pudo emparejar imagen/mascara sin ambiguedad en este inventario; queda para notebook 13."

if len(series_orientation_df) > 0:
    axial_candidates_df = series_orientation_df[series_orientation_df["orientation_candidate"].eq("axial_candidate")].copy()
    if len(axial_candidates_df) == 0:
        axial_candidates_df = series_orientation_df.copy()
        print("No se detectaron candidatos axiales explicitos; se usaran DICOM validos como fallback visual.")
else:
    axial_candidates_df = pd.DataFrame(columns=series_orientation_df.columns if "series_orientation_df" in globals() else [])
    print("No hay DICOM/IMA validos clasificados para visualizacion axial.")

for idx, row in enumerate(axial_candidates_df.head(5).itertuples(index=False), start=1):
    output_path = FIGURES_ROOT / f"E6_alkafri_axial_image_example_{idx:02d}.png"
    try:
        ds, pixels = read_dicom_pixels(row.file_path)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(normalize_percentile(pixels), cmap="gray")
        ax.set_title(f"Al-Kafri DICOM candidate {idx}\n{Path(row.relative_path).name}")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        axial_image_paths.append(str(output_path))
    except Exception as exc:
        visualization_errors.append({"file_path": getattr(row, "file_path", None), "error": repr(exc)})

if len(axial_image_paths) == 0:
    image_format_df = extracted_inventory_df[
        extracted_inventory_df["extension"].isin([".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"])
    ].copy()
    image_format_df = image_format_df[image_format_df.apply(is_visualization_raster_candidate, axis=1)].copy()
    print("No se pudo visualizar DICOM axial. Imagenes raster reales encontradas como alternativa:", len(image_format_df))
    for idx, row in enumerate(image_format_df.head(5).itertuples(index=False), start=1):
        output_path = FIGURES_ROOT / f"E6_alkafri_axial_image_example_{idx:02d}.png"
        try:
            image = np.asarray(Image.open(row.file_path))
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.imshow(normalize_percentile(image), cmap="gray")
            ax.set_title(f"Al-Kafri raster fallback {idx}\n{Path(row.relative_path).name}")
            ax.axis("off")
            fig.tight_layout()
            fig.savefig(output_path, dpi=160, bbox_inches="tight")
            plt.close(fig)
            axial_image_paths.append(str(output_path))
        except Exception as exc:
            visualization_errors.append({"file_path": getattr(row, "file_path", None), "error": repr(exc)})

mat_files_df = extracted_inventory_df[extracted_inventory_df["extension"].eq(".mat")].copy()
print("Archivos .mat detectados para analisis posterior:", len(mat_files_df))

print("Figuras axiales exportadas:")
for path in axial_image_paths:
    print(path)
if visualization_errors:
    print("Errores visualizacion:", visualization_errors[:5])
print("Overlay preliminar:", overlay_pairing_status)

Archivos .mat detectados para analisis posterior: 32
Figuras axiales exportadas:
/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_axial_image_example_01.png
/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_axial_image_example_02.png
/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_axial_image_example_03.png
/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_axial_image_example_04.png
/content/drive/MyDrive/PFI_MVP/figures/E6_alkafri_axial_image_example_05.png
Overlay preliminar: No se pudo emparejar imagen/mascara sin ambiguedad en este inventario; queda para notebook 13.


## Revision de Source_Code

In [28]:
SOURCE_KEYWORDS = ["dicom", "mask", "label", "ground", "disc", "thecal", "posterior", "l3", "l4", "l5"]
source_code_df = extracted_inventory_df[
    extracted_inventory_df["probable_source"].eq("source_code")
    & extracted_inventory_df["extension"].eq(".m")
].copy()

source_hits = []
for _, row in source_code_df.iterrows():
    path = Path(row["file_path"])
    try:
        text = path.read_text(encoding="utf-8", errors="ignore").lower()
    except Exception as exc:
        source_hits.append({
            "file_path": str(path),
            "relative_path": row["relative_path"],
            "read_ok": False,
            "read_error": repr(exc),
        })
        continue
    hit = {
        "file_path": str(path),
        "relative_path": row["relative_path"],
        "read_ok": True,
        "read_error": None,
    }
    for keyword in SOURCE_KEYWORDS:
        hit[f"count_{keyword}"] = int(text.count(keyword))
    source_hits.append(hit)

source_code_keyword_hits_df = pd.DataFrame(source_hits)
source_code_hits_csv_path = RESULTS_ROOT / "E6_alkafri_source_code_keyword_hits.csv"
source_code_keyword_hits_df.to_csv(source_code_hits_csv_path, index=False)

print("Archivos .m:", len(source_code_df))
print("Source code keyword hits:", source_code_hits_csv_path)
display(source_code_keyword_hits_df.head())

Archivos .m: 55
Source code keyword hits: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_source_code_keyword_hits.csv


,file_path,relative_path,read_ok,read_error,count_dicom,count_mask,count_label,count_ground,count_disc,count_thecal,count_posterior,count_l3,count_l4,count_l5
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/initialise.m,True,None,0,7,18,4,2,2,2,0,0,0
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/03_Extract_Best_Sli...,True,None,5,0,0,0,0,0,0,0,0,0
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/03_Extract_Best_Sli...,True,None,1,0,1,0,0,0,0,0,0,0
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/03_Extract_Best_Sli...,True,None,1,0,0,0,0,0,0,0,0,0
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,source_code/09_Source_Code/03_Extract_Best_Sli...,True,None,0,0,0,0,0,0,0,0,0,0


## Revision de Radiologists Notes

In [29]:
notes_df = extracted_inventory_df[extracted_inventory_df["probable_source"].eq("radiologist_notes")].copy()
notes_rows = []
for _, row in notes_df.iterrows():
    path = Path(row["file_path"])
    item = row.to_dict()
    item["preview"] = None
    item["preview_error"] = None
    if row["extension"] in {".txt", ".csv"}:
        try:
            item["preview"] = path.read_text(encoding="utf-8", errors="ignore")[:1000]
        except Exception as exc:
            item["preview_error"] = repr(exc)
    elif row["extension"] in {".xlsx", ".xls"}:
        try:
            preview_df = pd.read_excel(path, nrows=5)
            item["preview"] = preview_df.to_json(orient="records", force_ascii=False)
        except Exception as exc:
            item["preview_error"] = repr(exc)
    notes_rows.append(item)

radiologist_notes_inventory_df = pd.DataFrame(notes_rows)
radiologist_notes_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_radiologist_notes_inventory.csv"
radiologist_notes_inventory_df.to_csv(radiologist_notes_inventory_csv_path, index=False)

print("Radiologist notes files:", len(radiologist_notes_inventory_df))
print("Radiologist notes inventory:", radiologist_notes_inventory_csv_path)
display(radiologist_notes_inventory_df.head())

Radiologist notes files: 1
Radiologist notes inventory: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_radiologist_notes_inventory.csv


,file_path,relative_path,file_name,extension,parent_folder,size_bytes,probable_source,probable_type,is_nested,nested_source_zip,has_axial,has_sagittal,has_sag,has_mask,has_label,has_ground,has_gt,has_segmentation,has_disc,has_posterior,has_thecal,has_canal,has_intervertebral,has_l3,has_l4,has_l5,has_s1,preview,preview_error
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKA...,radiologist_notes/Radiologists Notes for Lumba...,Radiologists Report.xlsx,.xlsx,Radiologists Notes for Lumbar Spine MRI Dataset,38292,radiologist_notes,text_metadata,False,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,"[{""Patient ID"":1,""Clinician's Notes"":""L4-5: de...",None


## Reporte JSON

In [30]:
extension_counts = extracted_inventory_df["extension"].value_counts().to_dict() if "extension" in extracted_inventory_df.columns and len(extracted_inventory_df) else {}
probable_type_counts = extracted_inventory_df["probable_type"].value_counts().to_dict() if "probable_type" in extracted_inventory_df.columns and len(extracted_inventory_df) else {}
orientation_counts = (
    series_orientation_df["orientation_candidate"].value_counts().to_dict()
    if "orientation_candidate" in series_orientation_df.columns and len(series_orientation_df)
    else {}
)

nested_zips_detected = int(len(nested_zip_paths)) if "nested_zip_paths" in globals() else 0
nested_zips_extracted = int((nested_extraction_report_df["error"].isna() & (nested_extraction_report_df["extracted_file_count"] > 0)).sum()) if "nested_extraction_report_df" in globals() and len(nested_extraction_report_df) else 0
valid_dicom_files = int(len(valid_dicom_df)) if "valid_dicom_df" in globals() else 0
invalid_dicom_candidates = int(len(dicom_candidates_df) - valid_dicom_files) if "dicom_candidates_df" in globals() else 0

exports = {
    "zip_files": str(zip_files_csv_path),
    "zip_inventory": str(zip_inventory_csv_path),
    "extraction_report": str(extraction_report_csv_path),
    "nested_extraction_report": str(nested_extraction_report_csv_path),
    "extracted_file_inventory": str(extracted_inventory_csv_path),
    "folder_summary": str(folder_summary_csv_path),
    "extension_summary": str(extension_summary_csv_path),
    "dicom_metadata_sample": str(dicom_metadata_sample_csv_path),
    "dicom_reading_report": str(dicom_reading_report_csv_path),
    "series_orientation_candidates": str(series_orientation_csv_path),
    "ground_truth_inventory": str(ground_truth_inventory_csv_path),
    "source_code_keyword_hits": str(source_code_hits_csv_path),
    "radiologist_notes_inventory": str(radiologist_notes_inventory_csv_path),
    "axial_figures": axial_image_paths,
    "overlay_preliminary": overlay_preliminary_path,
}

if valid_dicom_files > 0 and len(ground_truth_inventory_df) > 0:
    recommendation_notebook_13 = "Construir preprocesamiento axial a partir de DICOM/IMA validos y ground truth emparejable."
elif int(extension_counts.get(".mat", 0)) > 0 or len(ground_truth_inventory_df) > 0:
    recommendation_notebook_13 = "Inspeccionar estructura de los .mat y ground truth para reconstruir pares imagen-mascara."
else:
    recommendation_notebook_13 = "Revisar estructura extraida y adaptar notebook 13 al formato real del dataset"

report = {
    "detected_zips": zip_files_df[["zip_name", "zip_path", "size_bytes", "size_human"]].to_dict(orient="records"),
    "expected_zips_all_detected": len(missing_expected) == 0,
    "nested_zips_detected": nested_zips_detected,
    "nested_zips_extracted": nested_zips_extracted,
    "total_extracted_files_after_nested": int(len(extracted_inventory_df)),
    "count_ima": int(extension_counts.get(".ima", 0)),
    "count_dcm": int(extension_counts.get(".dcm", 0)),
    "valid_dicom_files": valid_dicom_files,
    "invalid_dicom_candidates": invalid_dicom_candidates,
    "count_mat": int(extension_counts.get(".mat", 0)),
    "count_png": int(extension_counts.get(".png", 0)),
    "count_possible_masks_or_labels": int(probable_type_counts.get("mask_or_label_image", 0) + len(ground_truth_inventory_df)),
    "count_axial_candidate_files": int(orientation_counts.get("axial_candidate", 0)),
    "count_sagittal_candidate_files": int(orientation_counts.get("sagittal_candidate", 0)),
    "count_unknown_orientation_files": int(orientation_counts.get("unknown", 0)),
    "dicom_candidates_detected": int(len(dicom_candidates_df)) if "dicom_candidates_df" in globals() else 0,
    "dicom_orientation_classification_available": bool(valid_dicom_files > 0 and len(series_orientation_df) > 0),
    "dicom_read_success": bool(dicom_reading_report_df["read_ok"].fillna(False).any()) if "dicom_reading_report_df" in globals() and len(dicom_reading_report_df) else False,
    "visualized_at_least_one_axial_image": bool(len(axial_image_paths) > 0),
    "detected_ground_truth": bool(len(ground_truth_inventory_df) > 0),
    "paired_image_mask": bool(overlay_preliminary_path is not None),
    "pairing_status": overlay_pairing_status,
    "exports": exports,
    "recommendation_for_notebook_13": recommendation_notebook_13,
}

report_json_path = RESULTS_ROOT / "E6_alkafri_inventory_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte JSON:", report_json_path)
print(json.dumps(report, indent=2, ensure_ascii=False)[:5000])

Reporte JSON: /content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory/E6_alkafri_inventory_report.json
{
  "detected_zips": [
    {
      "zip_name": "Label Image Ground Truth Data for Lumbar Spine MRI Dataset.zip",
      "zip_path": "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips/Label Image Ground Truth Data for Lumbar Spine MRI Dataset.zip",
      "size_bytes": 1671165290,
      "size_human": "1.56 GB"
    },
    {
      "zip_name": "Radiologists Notes for Lumbar Spine MRI Dataset.zip",
      "zip_path": "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips/Radiologists Notes for Lumbar Spine MRI Dataset.zip",
      "size_bytes": 36027,
      "size_human": "35.18 KB"
    },
    {
      "zip_name": "Source_Code.zip",
      "zip_path": "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips/Source_Code.zip",
      "size_bytes": 1592126887,
      "size_human": "1.48 GB"
    },
    {
      "zip_name": "k57fr854j2-2.zip",
      "zip_path": "/content/drive/MyDriv

## Conclusion Markdown

In [31]:
zip_table_md = zip_files_df.to_markdown(index=False)
inventory_table_md = zip_inventory_df.to_markdown(index=False)
extension_table_md = extension_summary_df.head(30).to_markdown(index=False)
orientation_table_md = (
    series_orientation_df["orientation_candidate"].value_counts(dropna=False)
    .rename_axis("orientation_candidate")
    .reset_index(name="n")
    .to_markdown(index=False)
    if "orientation_candidate" in series_orientation_df.columns and len(series_orientation_df) else "Sin DICOM/IMA validos detectados o clasificables."
)
gt_table_md = (
    ground_truth_inventory_df[["relative_path", "extension", "probable_type"]].head(30).to_markdown(index=False)
    if len(ground_truth_inventory_df) and all(col in ground_truth_inventory_df.columns for col in ["relative_path", "extension", "probable_type"])
    else "No se detecto ground truth por inventario o keywords."
)

conclusion_text = f'''# Conclusion tecnica - E6 Inventario Al-Kafri/Sudirman

## Objetivo

Realizar inventario, descompresion controlada, analisis de estructura y lectura preliminar del dataset axial Al-Kafri/Sudirman, sin entrenar modelos.

## Por que se incorpora Al-Kafri/Sudirman

El spike axial sobre SPIDER mostro que las vistas axiales reconstruidas desde ese dataset pueden no ser visualmente adecuadas en varios casos por anisotropia, baja resolucion efectiva o geometria no nativa axial. Al-Kafri/Sudirman se incorpora como dataset complementario porque contiene cortes axiales nativos de RM lumbar y puede servir para el modulo axial del MVP.

## ZIPs originales y extraccion recursiva

Los ZIPs originales contenian ZIPs internos, por lo que fue necesaria una extraccion recursiva controlada dentro de `{NESTED_EXTRACTED_ROOT}`.

- ZIPs internos detectados: {report["nested_zips_detected"]}.
- ZIPs internos extraidos o ya disponibles: {report["nested_zips_extracted"]}.
- Total final de archivos despues de extraer internos: {report["total_extracted_files_after_nested"]}.

## Archivos cargados

{zip_table_md}

## Inventario interno de ZIPs

{inventory_table_md}

## Estructura y formatos encontrados

{extension_table_md}

## Lectura DICOM preliminar

- Candidatos DICOM/IMA detectados: {report["dicom_candidates_detected"]}.
- Archivos `.ima`: {report["count_ima"]}.
- Archivos `.dcm`: {report["count_dcm"]}.
- DICOM/IMA validos con tags medicos: {report["valid_dicom_files"]}.
- Candidatos descartados o invalidos: {report["invalid_dicom_candidates"]}.
- Clasificacion de orientacion disponible: {report["dicom_orientation_classification_available"]}.
- Lectura DICOM exitosa en al menos un archivo: {report["dicom_read_success"]}.

## Hallazgos sobre cortes axiales

{orientation_table_md}

Se exportaron figuras axiales preliminares cuando fue posible: {axial_image_paths}.

## Hallazgos sobre ground truth

{gt_table_md}

Ground truth detectado: {report["detected_ground_truth"]}.

Emparejamiento imagen/mascara: {overlay_pairing_status}

## Viabilidad preliminar

El dataset es viable para continuar si se confirma el formato real de imagenes, existen candidatos axiales nativos y el ground truth puede organizarse por paciente/serie/nivel. Si no se detectan DICOM/IMA validos pero aparecen `.mat` o raster labels, el siguiente paso debe inspeccionar esas estructuras para reconstruir pares imagen-mascara.

## Limitaciones

- No se entrenan modelos.
- No se asume estructura sin verificar.
- No se ejecuta codigo MATLAB.
- Las notas de radiologos se inventarian, pero no se usan para entrenamiento.
- El emparejamiento imagen/mascara queda pendiente si no hay una regla inequívoca.
- No se realizan diagnosticos ni mediciones clinicas.

## Siguiente paso recomendado

{recommendation_notebook_13}
'''

conclusion_path = DOCS_ROOT / "E6_alkafri_inventory_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion Markdown:", conclusion_path)

Conclusion Markdown: /content/drive/MyDrive/PFI_MVP/docs/E6_alkafri_inventory_conclusion.md


## Criterio de aceptacion

Este notebook cumple el objetivo si:

- Detecta los 4 ZIPs esperados.
- Inventaria el contenido de cada ZIP antes de extraer.
- Descomprime controladamente en `extracted`.
- Identifica imagenes DICOM/IMA.
- Intenta leer metadatos DICOM.
- Identifica candidatos axiales.
- Identifica ground truth.
- Genera visualizaciones axiales nativas cuando es posible.
- Exporta CSVs, JSON, Markdown y figuras.
- No entrena ningun modelo.